# Mutual Fund Facts Assistant: Technical Walkthrough

This notebook provides a comprehensive technical demonstration of the **Mutual Fund Facts Assistant**, a facts-only RAG (Retrieval-Augmented Generation) system designed to provide accurate, grounded information from official mutual fund documents.

--- 

## 1. Project Overview

The **Mutual Fund Facts Assistant** is a product concept designed for **Groww**. It serves as a specialized FAQ assistant for users seeking factual details about specific mutual fund schemes.

**Key Characteristics:**
- **Product Concept:** Designed as a feature for the Groww investment platform.
- **Target AMC:** Quant Mutual Fund.
- **Supported Schemes:** 
    1. Quant Small Cap Fund
    2. Quant Large Cap Fund
    3. Quant Flexi Cap Fund
    4. Quant Multi Cap Fund
    5. Quant PSU Fund
- **Strictly Facts-Only:** The system is engineered to retrieve and present factual information. It does **not** provide investment advice, recommendations, or performance predictions.
- **Non-Official Status:** This is a technical prototype and not an official product of Groww or Quant Mutual Fund.

## 2. Problem Statement

Investing in mutual funds requires access to critical factual data—such as minimum SIP amounts, expense ratios, exit loads, and fund managers. However, this information is often distributed across lengthy official documents (SIDs, Factsheets, and SAI) which can be daunting for a user to navigate manually.

**The Solution:**
A RAG-based assistant that indexes these official sources and retrieves precise answers. Unlike a general-purpose LLM, which may hallucinate or use outdated training data, this system ensures that every answer is grounded in **indexed official sources**, significantly increasing reliability and trust.

## 3. Scope

To maintain high factual accuracy and safety, the project has a strictly defined scope.

### ✅ Supported (In-Scope)
- **Investment Minimums:** Minimum SIP and lump-sum investment amounts.
- **Costs:** Expense Ratio / Total Expense Ratio (TER) and Exit Load conditions.
- **Governance:** Fund managers and investment objectives.
- **Risk & Benchmarking:** Riskometer levels and benchmark indices.
- **Operations:** Redemption timelines and account statement processes.
- **Taxation:** Capital gains and tax information available in the indexed sources.
- **Asset Allocation:** Permitted investment ranges for various asset classes.

### ❌ Outside Scope
- **Investment Advice:** No "buy/sell/hold" recommendations.
- **Fund Comparisons:** No "best fund" recommendations or performance rankings.
- **Predictions:** No return predictions or CAGR calculations.
- **Personalization:** No personalized portfolio advice or handling of sensitive PII (PAN, Aadhaar, etc.).
- **Unsupported Funds:** Only the five specified Quant Mutual Fund schemes are supported.

## 4. Supported Funds

The assistant is currently optimized for the following five schemes from Quant Mutual Fund:

1. **Quant Small Cap Fund**
2. **Quant Large Cap Fund**
3. **Quant Flexi Cap Fund**
4. **Quant Multi Cap Fund**
5. **Quant PSU Fund**

## 5. Official Data Sources

The knowledge base is built exclusively from **23 official public sources** to ensure the highest level of grounding. These include:

- **Scheme Information Documents (SID):** Detailed rules and objectives for each fund.
- **Scheme Presentations:** High-level overviews of fund strategies.
- **Quant Mutual Fund Factsheets:** Periodic data updates on fund holdings and risks.
- **Statement of Additional Information (SAI):** General AMC-level information.
- **Total Expense Ratio (TER) Documents:** Official pricing and charge details.
- **SEBI Investor Information:** Regulatory FAQs and investor guidelines.

## 6. Knowledge-Base Preparation

The pipeline transforms raw PDFs into a searchable vector database using the following steps:

1. **PDF-to-Markdown (Docling):** Official PDFs are converted to Markdown to preserve structural elements like tables and headings.
2. **Markdown Cleaning:** Unnecessary artifacts and formatting noise are removed to improve signal-to-noise ratio for the LLM.
3. **Heading-aware Chunking:** Instead of fixed-size splits, the system chunks text based on Markdown headings, ensuring that related information stays together.
4. **Metadata Enrichment:** Each chunk is tagged with the fund name, document type, and source file for precise filtering and citation.
5. **Embedding Generation:** Chunks are converted into high-dimensional vectors using the `all-MiniLM-L6-v2` model.
6. **Vector Storage (ChromaDB):** Embeddings and original text are stored in ChromaDB for efficient similarity search.

## 7. Metadata

Metadata is used to ensure the retrieval process is grounded and the answers are citable. The following fields are stored with every chunk:

- `fund_name`: The specific Quant fund the chunk refers to (e.g., "Quant Small Cap Fund").
- `document_type`: The nature of the source (e.g., "SID", "Factsheet", "TER").
- `source_file`: The filename of the original document used for citation mapping.
- `chunk_id`: A unique identifier for the segment within the document.

## 8. Embeddings and Vector Database

- **Embedding Model:** `all-MiniLM-L6-v2`. This model was chosen for its balance of performance and efficiency, providing strong semantic representations of financial text.
- **Vector Database:** **ChromaDB**. Used as the primary store for embeddings, allowing the system to perform semantic similarity searches to find the most relevant context for a user's query.

## 9. Hybrid Retrieval

To ensure the system doesn't miss precise terms (like "TER" or "SIP") while still understanding natural language, the project implements **Hybrid Retrieval**:

1. **Semantic Search:** Uses vector similarity to find chunks that are conceptually related to the query.
2. **BM25 (Keyword Search):** Uses traditional keyword-based scoring to find chunks containing exact terms.

By combining these two signals, the system captures both the *intent* and the *exact terminology* of the user's question, which is critical for factual financial retrieval.

## 10. Optional Cross-Encoder Reranking

To further refine the results, the system supports a reranking stage using the `cross-encoder/ms-marco-MiniLM-L-6-v2` model. 

**Implementation Detail:** Reranking is **optional** and controlled by the `USE_RERANKER` environment variable. This design choice was made to ensure the application remains functional in memory-constrained hosting environments where loading a second large model (the Cross-Encoder) might lead to crashes.

## 11. Groq LLM Generation

The final generation layer uses the **Groq API** with the `openai/gpt-oss-20b` model. Groq provides the extremely low latency required for a responsive user experience. 

The LLM is given a strict system prompt: **"Answer using ONLY the retrieved context."** This prevents the model from using general knowledge and forces it to stay grounded in the official documents.

## 12. Facts-Only Guardrails

Safety is paramount in financial applications. The system implements several hard guardrails:

- **Investment Advice Refusal:** Any query containing phrases like "should I invest" or "best fund" is intercepted and met with a standard refusal message directing the user to SEBI investor education.
- **Performance Guardrail:** Questions about returns, CAGR, or fund performance are blocked, as the assistant is designed for static facts, not dynamic performance analysis.
- **PII Protection:** The system uses regex patterns to detect and refuse to process sensitive information such as PAN, Aadhaar, or phone numbers.

## 13. Citation System

Every factual response is paired with a citation. The system maps the `source_file` from the retrieved chunk to a `SOURCE_URLS` dictionary, providing the user with:
1. The **Direct URL** to the official PDF source.
2. The **Last Updated Date** of that specific document (where available).

This transforms the assistant from a "black box" into a transparent tool that allows users to verify the information themselves.

## 14. Architecture Diagram

```mermaid
graph TD
    A[Official Public Sources] --> B[Docling / Markdown Processing]
    B --> C[Cleaning + Heading-aware Chunking]
    C --> D[Metadata Enrichment]
    D --> E[all-MiniLM-L6-v2 Embeddings]
    E --> F[ChromaDB]
    F --> G[Semantic Retrieval + BM25]
    G --> H[Optional Cross-Encoder Reranking]
    H --> I[Groq / openai-gpt-oss-20b]
    I --> J[Facts-only Answer + Official Citation]
```

## 15. Demonstration

Below is a live demonstration using the project's final RAG implementation. We will test several factual categories and the safety guardrails.

In [ ]:
import os
from search_rag_groq import ask_rag

# Ensure API key is set in environment
if not os.getenv("GROQ_API_KEY"):
    print("Please set the GROQ_API_KEY environment variable to run the demonstration.")
else:
    test_questions = [
        "What is the minimum SIP amount for Quant Small Cap Fund?",
        "Who manages Quant Large Cap Fund?",
        "What is the benchmark of Quant Flexi Cap Fund?",
        "What is the Riskometer level of Quant Multi Cap Fund?",
        "What is the exit load for Quant PSU Fund?",
        "What is the expense ratio of Quant Small Cap Fund?",
        "What are the capital gains tax rules for Quant Small Cap Fund?",
        "Should I invest in Quant Small Cap Fund?" # Guardrail Test
    ]
    
    for q in test_questions:
        print(f"\nQuestion: {q}")
        print("-" * 30)
        print(ask_rag(q))
        print("")
        print("=" * 70)

## 16. Example Output & Citation Verification

As seen in the demonstration above, every response follows a strict format:
- **Direct Answer:** Concise and grounded in context.
- **Source URL:** A clickable link to the official PDF.
- **Last Updated:** The date of the document to ensure the user knows the data's recency.

## 17. Evaluation

The system was rigorously evaluated using a set of 40 unseen questions. The evaluation focused on the accuracy of the retrieved facts and the model's ability to refuse out-of-scope queries.

**Key Findings:**
- **Grounding:** The system successfully retrieves specific values (like TER and Exit Loads) from the official documents.
- **Robustness:** The implementation of hybrid retrieval and intent-based overrides (for specific fund-manager and riskometer queries) significantly improved accuracy over a vanilla RAG approach.
- **Guardrails:** 100% of investment-advice and performance-related questions were correctly refused by the safety layer.

## 18. Known Limitations

- **Scheme Coverage:** Currently limited to five specific Quant Mutual Fund schemes.
- **Dynamic Data:** The system is not connected to live internal systems; it relies on the last indexed version of the official PDFs.
- **Scope:** No support for personalized financial advice or account-specific data.
- **Hosting Constraints:** Due to memory limits in certain deployment environments, reranking was made optional to prevent runtime crashes.

## 19. Product / Engineering Trade-offs

- **Accuracy vs. Memory:** By making the Cross-Encoder reranker optional, we traded a potential marginal gain in retrieval precision for guaranteed system stability across different hosting environments.
- **Strict Grounding vs. Breadth:** The system is intentionally restrictive. It would rather say "I could not find this information" than risk hallucinating a financial fact, prioritizing accuracy over the breadth of answers.
- **Hybrid Retrieval:** Added the complexity of BM25 to the pipeline to solve the issue where semantic search alone occasionally missed exact technical terms like "TER".

## 20. Disclaimer

**Facts-only. No investment advice.**

This assistant provides factual information from official public sources. It does not recommend mutual funds, predict returns, or provide personalized financial advice. Users should always verify current information using the linked official sources before making any financial decisions.